In [ ]:
### import necessary packages

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
### import data

data= pd.read_csv("./brca_metabric_clinical_data.tsv", sep="\t")

In [ ]:
### feature selection

# list of featires to keep
keep_columns=["Age at Diagnosis", 
              "Cancer Type Detailed", 
              "Inferred Menopausal State", 
              "Tumor Size",
              "Lymph nodes examined positive",
              "3-Gene classifier subtype",
              "Pam50 + Claudin-low subtype",
              "Integrative Cluster",
              "Overall Survival (Months)"]

# remove other features
data = data[keep_columns].copy()

# confirm removal
print(data.columns)

# remove sapmeles where Overall Survival is na
data = data.dropna(subset=["Overall Survival (Months)"])

# check shape
print(data.shape)

In [ ]:
### group rare cancer types together

top3 = data["Cancer Type Detailed"].value_counts().nlargest(3).index

data["Cancer Type Grouped"] = data["Cancer Type Detailed"].apply(
    lambda x: x if x in top3 else "Other")

print(data["Cancer Type Grouped"].value_counts())

In [ ]:
### discretize overall survival (months) into 2 groups

for index, row in data.iterrows():
    if row["Overall Survival (Months)"]>=60:
        data.loc[index, "Survival Duration"] = ">=5 years"
    else:
        data.loc[index, "Survival Duration"] = "<5 years"

In [ ]:
### split predictors and response

predictors=["Age at Diagnosis", 
              "Cancer Type Grouped", 
              "Inferred Menopausal State", 
              "Tumor Size",
              "Lymph nodes examined positive",
              "3-Gene classifier subtype",
              "Pam50 + Claudin-low subtype",
              "Integrative Cluster"]

x= data[predictors]
y=data["Survival Duration"]

In [ ]:
### split into test and train data 
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42)

In [ ]:
### statistical imputation

# check NaNs
print(x_train.isna().sum())
print(x_test.isna().sum())

# impute menopausal status based on age
mask = x_test["Inferred Menopausal State"].isna()
x_test.loc[mask, "Inferred Menopausal State"] = np.where(
    x_test.loc[mask, "Age at Diagnosis"] < 52,
    "Pre",
    "Post")

# impute tumor size from training median
tumor_size_median = x_train["Tumor Size"].median()
x_train["Tumor Size"] = x_train["Tumor Size"].fillna(tumor_size_median)
x_test["Tumor Size"] = x_test["Tumor Size"].fillna(tumor_size_median)

# impute lymph nodes from training median
lymph_node_median = x_train["Lymph nodes examined positive"].median()
x_train["Lymph nodes examined positive"] = x_train["Lymph nodes examined positive"].fillna(lymph_node_median)
x_test["Lymph nodes examined positive"] = x_test["Lymph nodes examined positive"].fillna(lymph_node_median)

# impute 3-gene classifier using knn
three_gene_mode = x_train["3-Gene classifier subtype"].mode()[0]
x_train["3-Gene classifier subtype"] = x_train["3-Gene classifier subtype"].fillna(three_gene_mode)
x_test["3-Gene classifier subtype"] = x_test["3-Gene classifier subtype"].fillna(three_gene_mode)

# impute Pam50 using knn
pam50_mode = x_train["Pam50 + Claudin-low subtype"].mode()[0]
x_train["Pam50 + Claudin-low subtype"] = x_train["Pam50 + Claudin-low subtype"].fillna(pam50_mode)
x_test["Pam50 + Claudin-low subtype"] = x_test["Pam50 + Claudin-low subtype"].fillna(pam50_mode)

# impute IntCluster using knn
int_cluster_mode = x_train["Integrative Cluster"].mode()[0]
x_train["Integrative Cluster"] = x_train["Integrative Cluster"].fillna(int_cluster_mode)
x_test["Integrative Cluster"] = x_test["Integrative Cluster"].fillna(int_cluster_mode)

In [ ]:
### confirm NaNs are removed

print(x_train.isna().sum())
print(x_test.isna().sum())

In [ ]:
### seperate x train and x test for each model

# model 1- baseline
x1=["Age at Diagnosis", 
    "Cancer Type Grouped", 
    "Inferred Menopausal State", 
    "Tumor Size",
    "Lymph nodes examined positive"]
x1_train= x_train[x1]
x1_test= x_test[x1]

# model 2- baseline + 3-gene
x2= x1 + ["3-Gene classifier subtype"]
x2_train= x_train[x2]
x2_test= x_test[x2]

# model 3- baseline + Pam 50
x3= x1 + ["Pam50 + Claudin-low subtype"]
x3_train= x_train[x3]
x3_test= x_test[x3]

# model 4 - baseline + integrative clusters
x4= x1 + ["Integrative Cluster"]
x4_train= x_train[x4]
x4_test= x_test[x4]

# model 5- 3-gene
x5= ["3-Gene classifier subtype"]
x5_train= x_train[x5]
x5_test= x_test[x5]

# model 6- Pam 50 
x6= ["Pam50 + Claudin-low subtype"]
x6_train= x_train[x6]
x6_test= x_test[x6]

# model 7- Integrative clusters
x7= ["Integrative Cluster"]
x7_train= x_train[x7]
x7_test= x_test[x7]

In [ ]:
###  Remove NC (not classified) from x3 and x6

# Training set
mask_train = x3_train["Pam50 + Claudin-low subtype"] != "NC"
x3_train = x3_train[mask_train]
y3_train = y_train[mask_train]

mask_train = x6_train["Pam50 + Claudin-low subtype"] != "NC"
x6_train = x6_train[mask_train]
y6_train = y_train[mask_train]

# Test set
mask_test = x3_test["Pam50 + Claudin-low subtype"] != "NC"
x3_test = x3_test[mask_test]
y3_test = y_test[mask_test]

mask_test = x6_test["Pam50 + Claudin-low subtype"] != "NC"
x6_test = x6_test[mask_test]
y6_test = y_test[mask_test]

In [ ]:
### One hot encode

x1_train = pd.get_dummies(x1_train)
x1_test = pd.get_dummies(x1_test)

x2_train = pd.get_dummies(x2_train)
x2_test = pd.get_dummies(x2_test)

x3_train = pd.get_dummies(x3_train)
x3_test = pd.get_dummies(x3_test)

x4_train = pd.get_dummies(x4_train)
x4_test = pd.get_dummies(x4_test)

x5_train = pd.get_dummies(x5_train)
x5_test = pd.get_dummies(x5_test)

x6_train = pd.get_dummies(x6_train)
x6_test = pd.get_dummies(x6_test)

x7_train = pd.get_dummies(x7_train)
x7_test = pd.get_dummies(x7_test)

In [ ]:
### establish RandomForest models

# model 1
rf1 = RandomForestClassifier(
    n_estimators=400,
    max_depth=5,
    min_samples_leaf=5,
    random_state=42,
    class_weight='balanced_subsample')

# model 2
rf2 = RandomForestClassifier(
    n_estimators=400,
    max_depth=5,
    min_samples_leaf=5,
    random_state=42,
    class_weight='balanced_subsample')

# model 3
rf3 = RandomForestClassifier(
    n_estimators=400,
    max_depth=5,
    min_samples_leaf=5,
    random_state=42,
    class_weight='balanced_subsample')

# model 4
rf4 = RandomForestClassifier(
    n_estimators=400,
    max_depth=5,
    min_samples_leaf=5,
    random_state=42,
    class_weight='balanced_subsample')

# model 5
rf5 = RandomForestClassifier(
    n_estimators=400,
    max_depth=5,
    min_samples_leaf=5,
    random_state=42,
    class_weight='balanced_subsample')

# model 6
rf6 = RandomForestClassifier(
    n_estimators=400,
    max_depth=5,
    min_samples_leaf=5,
    random_state=42,
    class_weight='balanced_subsample')

# model 7
rf7 = RandomForestClassifier(
    n_estimators=400,
    max_depth=5,
    min_samples_leaf=5,
    random_state=42,
    class_weight='balanced_subsample')


In [ ]:
### fit models 

# model 1
rf1.fit(x1_train, y_train)

# model 2
rf2.fit(x2_train, y_train)

# model 3
rf3.fit(x3_train, y3_train)

# model 4
rf4.fit(x4_train, y_train)

# model 5
rf5.fit(x5_train, y_train)

# model 6
rf6.fit(x6_train, y6_train)

# model 7
rf7.fit(x7_train, y_train)

In [ ]:
### make predictions

# model 1
y1_pred = rf1.predict(x1_test)

# model 2
y2_pred = rf2.predict(x2_test)

# model 3
y3_pred = rf3.predict(x3_test)

# model 4
y4_pred = rf4.predict(x4_test)

# model 5
y5_pred = rf5.predict(x5_test)

# model 6
y6_pred = rf6.predict(x6_test)

# model 7
y7_pred = rf7.predict(x7_test)

In [ ]:
### evaluate models

# model 1
print("Accuracy:", accuracy_score(y_test, y1_pred))
print(classification_report(y_test, y1_pred))

# model 2
print("Accuracy:", accuracy_score(y_test, y2_pred))
print(classification_report(y_test, y2_pred))

# model 3
print("Accuracy:", accuracy_score(y3_test, y3_pred))
print(classification_report(y3_test, y3_pred))

# model 4
print("Accuracy:", accuracy_score(y_test, y4_pred))
print(classification_report(y_test, y4_pred))

# model 5
print("Accuracy:", accuracy_score(y_test, y5_pred))
print(classification_report(y_test, y5_pred))

# model 6
print("Accuracy:", accuracy_score(y6_test, y6_pred))
print(classification_report(y6_test, y6_pred))

# model 7
print("Accuracy:", accuracy_score(y_test, y7_pred))
print(classification_report(y_test, y7_pred))

In [ ]:
### plot most important features

# model 1
importances1 = pd.Series(rf1.feature_importances_, index=x1_train.columns)
top_features1 = importances1.sort_values(ascending=False).head(15)

plt.figure(figsize=(8,5))
top_features1.plot(kind="bar")
plt.title("Top Feature Importances - RF1")
plt.ylabel("Importance")
plt.show()

# model 2
importances2 = pd.Series(rf2.feature_importances_, index=x2_train.columns)
top_features2 = importances2.sort_values(ascending=False).head(15)

plt.figure(figsize=(8,5))
top_features2.plot(kind="bar")
plt.title("Top Feature Importances - RF2")
plt.ylabel("Importance")
plt.show()

# model 3
importances3 = pd.Series(rf3.feature_importances_, index=x3_train.columns)
top_features3 = importances3.sort_values(ascending=False).head(15)

plt.figure(figsize=(8,5))
top_features3.plot(kind="bar")
plt.title("Top Feature Importances - RF3")
plt.ylabel("Importance")
plt.show()

# model 4
importances4 = pd.Series(rf4.feature_importances_, index=x4_train.columns)
top_features4 = importances4.sort_values(ascending=False).head(15)

plt.figure(figsize=(8,5))
top_features4.plot(kind="bar")
plt.title("Top Feature Importances - RF4")
plt.ylabel("Importance")
plt.show()